In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras

In [2]:
DATA_PATH = "../data/multipliers.csv"
WINDOW = 20          
HIDDEN = 32          
EPOCHS = 30
BATCH = 256
LR = 1e-3
SEED = 42
TRAIN_RATIO, VAL_RATIO = 0.8, 0.1 

In [3]:
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [4]:
series = pd.read_csv(DATA_PATH)["Multiplier"].astype(np.float64).values
n = len(series)
train_end = int(n * TRAIN_RATIO)
val_end = int(n * (TRAIN_RATIO + VAL_RATIO))

In [5]:
cap = np.quantile(series[:train_end], 0.99)
series_clipped = np.minimum(series, cap)

In [6]:
log_series = np.log(series_clipped)
mu = log_series[:train_end].mean()
sd = log_series[:train_end].std()
scaled = ((log_series - mu) / sd).astype(np.float32)

In [7]:
def make_windows(arr, w):
    X = np.lib.stride_tricks.sliding_window_view(arr, w)[:-1]  # (N-w, w)
    y = arr[w:]                                                # (N-w,)
    return X[..., np.newaxis], y

In [8]:
X_all, y_all = make_windows(scaled, WINDOW)


In [9]:
t_train_end = train_end - WINDOW
t_val_end = val_end - WINDOW
X_tr, y_tr = X_all[:t_train_end], y_all[:t_train_end]
X_va, y_va = X_all[t_train_end:t_val_end], y_all[t_train_end:t_val_end]
X_te, y_te = X_all[t_val_end:], y_all[t_val_end:]

In [10]:
model = keras.Sequential([
    keras.Input(shape=(WINDOW, 1)),
    keras.layers.SimpleRNN(HIDDEN),
    keras.layers.Dense(1),
])

In [11]:
model.compile(optimizer=keras.optimizers.Adam(LR), loss="mse", metrics=["mae"])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)               │ (None, 32)                  │           1,088 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,121 (4.38 KB)

 Trainable params: 1,121 (4.38 KB)

 Non-trainable params: 0 (0.00 B)

In [12]:
early = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=4, restore_best_weights=True
)

In [13]:
history = model.fit(
    X_tr, y_tr,
    validation_data=(X_va, y_va),
    epochs=EPOCHS,
    batch_size=BATCH,
    shuffle=True,         
    callbacks=[early],
    verbose=2,
)

Epoch 1/30
2289/2289 - 37s - 16ms/step - loss: 1.0035 - mae: 0.7607 - val_loss: 1.0073 - val_mae: 0.7684
Epoch 2/30
2289/2289 - 40s - 18ms/step - loss: 1.0009 - mae: 0.7597 - val_loss: 1.0068 - val_mae: 0.7674
Epoch 3/30
2289/2289 - 42s - 18ms/step - loss: 1.0006 - mae: 0.7596 - val_loss: 1.0066 - val_mae: 0.7669
Epoch 4/30
2289/2289 - 32s - 14ms/step - loss: 1.0004 - mae: 0.7596 - val_loss: 1.0065 - val_mae: 0.7666
Epoch 5/30
2289/2289 - 42s - 18ms/step - loss: 1.0003 - mae: 0.7595 - val_loss: 1.0063 - val_mae: 0.7662
Epoch 6/30
2289/2289 - 42s - 18ms/step - loss: 1.0002 - mae: 0.7595 - val_loss: 1.0062 - val_mae: 0.7657
Epoch 7/30
2289/2289 - 33s - 14ms/step - loss: 1.0002 - mae: 0.7595 - val_loss: 1.0061 - val_mae: 0.7652
Epoch 8/30
2289/2289 - 34s - 15ms/step - loss: 1.0001 - mae: 0.7594 - val_loss: 1.0060 - val_mae: 0.7650
Epoch 9/30
2289/2289 - 41s - 18ms/step - loss: 1.0001 - mae: 0.7594 - val_loss: 1.0059 - val_mae: 0.7650
Epoch 10/30
2289/2289 - 34s - 15ms/step - loss: 1.0001 

In [14]:
def inverse(z):
    return np.exp(z * sd + mu)
 
y_true = inverse(y_te)                 # test gerçek değerleri (kırpılmış)
pred_rnn = inverse(model.predict(X_te, verbose=0).ravel())

In [15]:
last_vals = inverse(X_te[:, -1, 0])

In [16]:
train_median = np.median(np.exp(log_series[:train_end]))
pred_median = np.full_like(y_true, train_median)

In [17]:
def report(name, pred):
    mae = np.mean(np.abs(pred - y_true))
    rmse = np.sqrt(np.mean((pred - y_true) ** 2))
    print(f"{name:<22} MAE={mae:.4f}  RMSE={rmse:.4f}")

In [18]:
print("\n--- Regresyon (test) ---")
report("SimpleRNN", pred_rnn)
report("Baseline: son değer", last_vals)
report("Baseline: medyan", pred_median)


--- Regresyon (test) ---
SimpleRNN              MAE=4.1084  RMSE=12.5418
Baseline: son değer    MAE=6.8486  RMSE=17.3142
Baseline: medyan       MAE=4.0044  RMSE=12.7123


In [19]:
THRESH = 2.0
y_cls = (y_true >= THRESH).astype(int)
p_cls = (pred_rnn >= THRESH).astype(int)

In [20]:
majority = max(y_cls.mean(), 1 - y_cls.mean())
acc = (p_cls == y_cls).mean()
print("\n--- Sınıflandırma (>= 2.0) ---")
print(f"Pozitif (>=2) oranı (test): {y_cls.mean():.3f}")
print(f"Çoğunluk sınıfı baseline acc: {majority:.3f}")
print(f"SimpleRNN accuracy:           {acc:.3f}")
print(f"Tahmin edilen pozitif oranı:  {p_cls.mean():.3f}")


--- Sınıflandırma (>= 2.0) ---
Pozitif (>=2) oranı (test): 0.483
Çoğunluk sınıfı baseline acc: 0.517
SimpleRNN accuracy:           0.483
Tahmin edilen pozitif oranı:  1.000


In [22]:
model.save("../model/aviator_game.keras")